In [3]:
%pip install notebook jupyterlab
%pip install pandas ipython
%pip install google-colab

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
ERROR: Could not find a version that satisfies the requirement google-colab (from versions: none)
ERROR: No matching distribution found for google-colab
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
# @title 📊 Aurora Finance Capital Allocation Dashboard {display-mode: "form"}

import pandas as pd
import json
from IPython.display import HTML
# from google.colab import output

# Read data from CSV output file
df_merged = pd.read_csv('/Users/himanshmac/MLFin/finance-budgeting/outputs/module1_ranked_projects.csv')

# Parse currency columns (remove $ and commas)
for col in ['Investment_Cost', 'NPV', 'Expected_Value', 'Predicted_NPV', 'Risk_Adjusted_Return']:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].astype(str).str.replace('$', '').str.replace(',', '').astype(float)

# Parse percentage columns
for col in ['Predicted_Success_Probability', 'Historical_ROI']:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].astype(str).str.replace('%', '').astype(float) / 100

# Create IRR column from Historical_ROI if not exists
if 'IRR' not in df_merged.columns:
    df_merged['IRR'] = df_merged['Historical_ROI']

# Create Risk_Adjusted_EV column if not exists
if 'Risk_Adjusted_EV' not in df_merged.columns:
    df_merged['Risk_Adjusted_EV'] = df_merged['Expected_Value'] * df_merged['Predicted_Success_Probability']

# Create Executive_Summary if not exists
if 'Executive_Summary' not in df_merged.columns:
    df_merged['Executive_Summary'] = df_merged.apply(
        lambda row: f"{row['Recommendation']}: NPV ${row['NPV']:,.0f}, Success Prob {row['Predicted_Success_Probability']:.1%}",
        axis=1
    )

# Convert to JSON for injection
projects_json = df_merged.to_json(orient='records')

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

# output.register_callback('report_js_error', _report_js_error)

html_code = """
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f4f6f8; margin: 0; padding: 20px; color: #333; }
        .dashboard-container { display: flex; flex-direction: column; gap: 20px; }
        .controls { background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); display: flex; align-items: center; gap: 15px; }
        .kpi-row { display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 20px; }
        .kpi-card { background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); text-align: center; }
        .kpi-value { font-size: 24px; font-weight: bold; color: #1a237e; margin: 10px 0; }
        .kpi-label { font-size: 14px; color: #666; text-transform: uppercase; }
        .chart-row { display: grid; grid-template-columns: 2fr 1fr; gap: 20px; }
        .valuation-row { display: grid; grid-template-columns: 1fr; gap: 20px; }
        .card { background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); display: flex; flex-direction: column; min-height: 350px; }
        .card h3 { margin-top: 0; font-size: 16px; color: #1a237e; border-bottom: 1px solid #eee; padding-bottom: 10px; }
        .canvas-wrapper { position: relative; flex-grow: 1; min-height: 0; }
        .table-container { background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); overflow-x: auto; }
        table { width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 13px; }
        th { background: #1a237e; color: white; text-align: left; padding: 12px; position: sticky; top: 0; }
        td { padding: 10px; border-bottom: 1px solid #eee; }
        .rec-fund { color: #2e7d32; font-weight: bold; }
        .rec-review { color: #ef6c00; font-weight: bold; }
        .rec-reject { color: #c62828; font-weight: bold; }
        select { padding: 8px; border-radius: 4px; border: 1px solid #ddd; outline: none; }
    </style>
</head>
<body>
    <div class="dashboard-container">
        <div class="controls">
            <label style="font-weight:bold">Strategic Filter:</label>
            <select id="recFilter" onchange="updateDashboard()">
                <option value="All">All Projects</option>
                <option value="Fund">Fund Only</option>
                <option value="Review">Review Required</option>
                <option value="Reject">Rejected</option>
            </select>
        </div>

        <div class="kpi-row">
            <div class="kpi-card"><div class="kpi-label">Total Capex</div><div id="kpi-capex" class="kpi-value">$0</div></div>
            <div class="kpi-card"><div class="kpi-label">Portfolio NPV</div><div id="kpi-npv" class="kpi-value">$0</div></div>
            <div class="kpi-card"><div class="kpi-label">Avg IRR</div><div id="kpi-irr" class="kpi-value">0%</div></div>
            <div class="kpi-card"><div class="kpi-label">Project Count</div><div id="kpi-count" class="kpi-value">0</div></div>
        </div>

        <div class="chart-row">
            <div class="card">
                <h3>IRR Performance by Project</h3>
                <div class="canvas-wrapper"><canvas id="irrChart"></canvas></div>
            </div>
            <div class="card">
                <h3>Capex by Department</h3>
                <div class="canvas-wrapper"><canvas id="deptChart"></canvas></div>
            </div>
        </div>

        <div class="valuation-row">
            <div class="card">
                <h3>Valuation Subplot: Expected Value vs Risk-Adjusted EV</h3>
                <div class="canvas-wrapper"><canvas id="valChart"></canvas></div>
            </div>
        </div>

        <div class="table-container">
            <h3 style="color:#1a237e; margin-top:0">Project Summary & Heuristics</h3>
            <table id="projectTable">
                <thead>
                    <tr>
                        <th>Rank</th><th>ID</th><th>Dept</th><th>Capex</th><th>NPV</th><th>IRR</th><th>Decision</th><th>Summary</th>
                    </tr>
                </thead>
                <tbody></tbody>
            </table>
        </div>
    </div>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        const rawData = DATA_PLACEHOLDER;
        let irrChart, deptChart, valChart;

        function formatCurrency(val) {
            return new Intl.NumberFormat('en-US', { style: 'currency', currency: 'USD', maximumFractionDigits: 0 }).format(val);
        }

        function initCharts() {
            const ctxIrr = document.getElementById('irrChart').getContext('2d');
            irrChart = new Chart(ctxIrr, {
                type: 'bar',
                data: { labels: [], datasets: [{ label: 'IRR %', data: [], backgroundColor: '#3949ab' }] },
                options: { responsive: true, maintainAspectRatio: false, scales: { y: { beginAtZero: true, ticks: { callback: v => v + '%' } } } }
            });

            const ctxDept = document.getElementById('deptChart').getContext('2d');
            deptChart = new Chart(ctxDept, {
                type: 'doughnut',
                data: { labels: [], datasets: [{ data: [], backgroundColor: ['#1a237e', '#3949ab', '#7986cb', '#c5cae9'] }] },
                options: { responsive: true, maintainAspectRatio: false, plugins: { legend: { position: 'bottom' } } }
            });

            const ctxVal = document.getElementById('valChart').getContext('2d');
            valChart = new Chart(ctxVal, {
                type: 'bar',
                data: { labels: [], datasets: [
                    { label: 'Expected Value', data: [], backgroundColor: '#3949ab' },
                    { label: 'Risk-Adjusted EV', data: [], backgroundColor: '#ef5350' }
                ] },
                options: { responsive: true, maintainAspectRatio: false, scales: { y: { beginAtZero: true } } }
            });
        }

        function updateDashboard() {
            const filter = document.getElementById('recFilter').value;
            const filtered = filter === 'All' ? rawData : rawData.filter(d => d.Recommendation === filter);

            // Update KPIs
            const totalCapex = filtered.reduce((a, b) => a + b.Investment_Cost, 0);
            const totalNpv = filtered.reduce((a, b) => a + b.NPV, 0);
            const avgIrr = filtered.length ? (filtered.reduce((a, b) => a + b.IRR, 0) / filtered.length) * 100 : 0;

            document.getElementById('kpi-capex').innerText = formatCurrency(totalCapex);
            document.getElementById('kpi-npv').innerText = formatCurrency(totalNpv);
            document.getElementById('kpi-irr').innerText = avgIrr.toFixed(1) + '%';
            document.getElementById('kpi-count').innerText = filtered.length;

            // Update Charts
            irrChart.data.labels = filtered.map(d => 'P' + d.Project_ID);
            irrChart.data.datasets[0].data = filtered.map(d => (d.IRR * 100).toFixed(2));
            irrChart.update();

            const depts = [...new Set(filtered.map(d => d.Department))];
            deptChart.data.labels = depts;
            deptChart.data.datasets[0].data = depts.map(dept => filtered.filter(d => d.Department === dept).reduce((a, b) => a + b.Investment_Cost, 0));
            deptChart.update();

            valChart.data.labels = filtered.map(d => 'P' + d.Project_ID);
            valChart.data.datasets[0].data = filtered.map(d => d.Expected_Value);
            valChart.data.datasets[1].data = filtered.map(d => d.Risk_Adjusted_EV);
            valChart.update();

            // Update Table
            const tbody = document.querySelector('#projectTable tbody');
            tbody.innerHTML = '';
            filtered.forEach(d => {
                const row = `<tr>
                    <td>${d.Rank}</td>
                    <td>${d.Project_ID}</td>
                    <td>${d.Department}</td>
                    <td>${formatCurrency(d.Investment_Cost)}</td>
                    <td>${formatCurrency(d.NPV)}</td>
                    <td>${(d.IRR * 100).toFixed(1)}%</td>
                    <td class="rec-${d.Recommendation.toLowerCase()}">${d.Recommendation}</td>
                    <td style="font-size:11px; max-width:300px">${d.Executive_Summary}</td>
                </tr>`;
                tbody.innerHTML += row;
            });
        }

        initCharts();
        updateDashboard();
    </script>
</body>
</html>
""".replace('DATA_PLACEHOLDER', projects_json)

display(HTML(html_code))

Rank,ID,Dept,Capex,NPV,IRR,Decision,Summary
